# M3 — The benchmark: 120 test questions + correct answers

**Goal of this step:** build the "exam" that every version (A–E) will take.

**Analogy:** before teaching a class, a teacher writes the exam **and the answer key**, then locks them in a drawer.
If you wrote the exam *after* seeing how students study, you could (even by accident) make it easy for your
favourite method. That's why the benchmark is built and **frozen** before versions B–E exist.

**What's in the exam**
| Part | How many | What it tests |
|---|---|---|
| Easy | 40 | one table, no joins |
| Medium | 40 | two tables must be **joined** |
| Tricky | 40 | a **look-alike** table is easy to pick by mistake |
| Conversations | 20 × 3 turns | follow-ups like *"now split that by age"* (memory) |

Split: **40 practice** questions (you may look at these while building) and **80 test** questions (only used for
the final scores). Conversations: 5 practice / 15 test.

**Files in this step**
| File | What it does |
|---|---|
| `benchmark/make_benchmark.py` | 30 question templates + 4 conversation templates; fills them with real IDs |
| `benchmark/questions.jsonl` | the 120 questions (one per line) |
| `benchmark/conversations.jsonl` | the 20 conversations |
| `benchmark/gold/*.sql` | the correct SQL for every question / turn (180 files) |
| `benchmark/check_gold.py` | runs every gold SQL, checks the answers, saves them to `gold_results.jsonl`, can freeze |
| `benchmark/cross_check.py` | recomputes some answers from the **raw** data a second way |

**Done when:** every gold SQL runs, and the practice/test split is frozen ✅ (freezing is your last step, after review)

In [1]:
import sys, json
sys.path.append("../benchmark")

def load_jsonl(path):
    return [json.loads(line) for line in open(path)]

## 1. How a question is made: template + real IDs
One **template** = one kind of question, written once, with its correct SQL. Real IDs are then picked from the
data. Only IDs with enough data are picked (e.g. campaigns with ≥ 100 impressions every day), and
**no ID is used twice**. A fixed random seed (42) means the same benchmark comes out every time.

Here's one template, T2. Its trap: adding up the daily unique people counts the same person several times.

In [2]:
from make_benchmark import TEMPLATES
t2 = [t for t in TEMPLATES if t["key"] == "T2"][0]
for k, v in t2.items():
    print(f"{k:10s}: {v}")

key       : T2
tier      : tricky
needs     : ['campaign']
tables    : ['reach_campaign_total']
trap      : adding up reach_campaign_daily counts people twice
questions : ['How many unique people saw campaign {campaign} from 6 to 12 May?', 'Total reach (unique users) of campaign {campaign} over the final period?']
sql       : SELECT unique_users FROM reach_campaign_total WHERE campaign_id = {campaign}


## 2. Make the benchmark

In [3]:
from make_benchmark import make_benchmark
make_benchmark()

Wrote 120 questions and 20 conversations to /Users/anshikapuneet/Documents/Claude_history/capstone_project/benchmark


## 3. Look at some questions
One example from each template. `values` are the IDs that were picked.

In [4]:
questions = load_jsonl("../benchmark/questions.jsonl")
seen = set()
for q in questions:
    if q["template"] not in seen:
        seen.add(q["template"])
        print(f"{q['id']}  {q['tier']:6s} {q['template']:3s} {q['split']:8s}  {q['question']}")

q001  easy   E1  practice  What was the CTR of campaign 279732 on 6 May 2017?
q005  easy   E2  test      How many clicks did ad 674231 get on 6 May 2017?
q009  easy   E3  test      How many impressions did brand 381566 have on 6 May 2017?
q013  easy   E4  practice  Which product category got the most clicks on 6 May 2017, and how many?
q017  easy   E5  test      Which advertiser runs campaign 283458?
q021  easy   E6  test      How many ads does campaign 116859 have?
q025  easy   E7  practice  What is the item price for ad 705950?
q029  easy   E8  practice  What was the estimated value for advertiser 38817 on 11 May 2017?
q033  easy   E9  test      Which ad placement had the higher CTR on 8 May 2017?
q037  easy   E10 practice  How many campaigns does advertiser 22226 run?
q041  medium M1  practice  How many clicks did advertiser 150722's ads in category 6183 get yesterday?
q045  medium M2  test      What was the CTR among heavy shoppers (shopping depth 3) for campaign 350441 on 6 May 20

And here's one full record, with its gold SQL:

In [5]:
q = questions[80]   # a tricky one
print(json.dumps(q, indent=2))
print("\nGold SQL:\n" + open("../benchmark/" + q["gold_sql"]).read())

{
  "id": "q081",
  "tier": "tricky",
  "template": "T1",
  "question": "What was campaign 143747's CTR yesterday, excluding suspicious clicks?",
  "gold_tables": [
    "agg_campaign_daily_valid"
  ],
  "trap": "all-traffic table agg_campaign_daily",
  "values": {
    "campaign_suspicious": 143747
  },
  "split": "test",
  "gold_sql": "gold/q081.sql"
}

Gold SQL:
SELECT ctr FROM agg_campaign_daily_valid WHERE campaign_id = 143747 AND event_date = '2017-05-12'



## 4. Run every gold SQL and check the answers
`check_gold()` runs all 180 gold SQLs (120 questions + 20 × 3 conversation turns) and flags:
- SQL errors
- empty answers or NULL answers
- **ties** on "top 1" questions (if two brands share first place, there's no single right answer)

In [6]:
from check_gold import check_gold
check_gold()

Questions: 120  {('easy', 'practice'): 13, ('easy', 'test'): 27, ('medium', 'practice'): 13, ('medium', 'test'): 27, ('tricky', 'test'): 26, ('tricky', 'practice'): 14}
Conversations: 20  {'practice': 5, 'test': 15}
Duplicate questions: 0
Gold SQL run: 180   problems: 0
Fingerprint: 61e282210933c887


True

In [7]:
gold = {r["id"]: r["rows"] for r in load_jsonl("../benchmark/gold_results.jsonl")}
for qid in ["q001", "q053", "q081", "q101", "q109"]:
    q = [x for x in questions if x["id"] == qid][0]
    print(f"{qid}: {q['question']}\n      -> {gold[qid]}\n")

q001: What was the CTR of campaign 279732 on 6 May 2017?
      -> [[0.058394160583941604]]

q053: Break down advertiser 56209's estimated value yesterday by product category.
      -> [[856, 2967.0], [6183, 4214.0]]

q081: What was campaign 143747's CTR yesterday, excluding suspicious clicks?
      -> [[0.056338028169014086]]

q101: How many clicks did campaign 319897 get in the week starting Monday 8 May?
      -> [[533, 5]]

q109: What was the CTR for campaign 101417 on 7 May 2017, split by gender?
      -> [['female', 0.07692307692307693], ['male', 0.0967741935483871], ['unknown', 0.05555555555555555]]



## 5. Double-check from the raw data (a second, independent way)
The gold SQL reads the tables **built in M1**. If the M1 build had a bug, the "correct" answers would be wrong
too, and nobody would notice. So `cross_check.py` recomputes 28 answers straight from `raw_sample` + `ad_feature`
(including converting the time zone again) and compares.

In [8]:
from cross_check import cross_check
cross_check()

OK   q001 E1  gold=0.058394160583941604  raw=0.058394160583941604
OK   q002 E1  gold=0.06293706293706294  raw=0.06293706293706294
OK   q003 E1  gold=0.062111801242236024  raw=0.062111801242236024
OK   q004 E1  gold=0.04477611940298507  raw=0.04477611940298507
OK   q005 E2  gold=4  raw=4
OK   q006 E2  gold=12  raw=12
OK   q007 E2  gold=11  raw=11
OK   q008 E2  gold=13  raw=13


OK   q009 E3  gold=1244  raw=1244
OK   q010 E3  gold=449  raw=449
OK   q011 E3  gold=397  raw=397


OK   q012 E3  gold=745  raw=745
OK   q025 E7  gold=336.0  raw=336.0
OK   q026 E7  gold=339.0  raw=339.0
OK   q027 E7  gold=1590.0  raw=1590.0
OK   q028 E7  gold=128.0  raw=128.0
OK   q085 T2  gold=1821  raw=1821
OK   q086 T2  gold=1411  raw=1411
OK   q087 T2  gold=861  raw=861
OK   q088 T2  gold=1292  raw=1292
OK   q089 T3  gold=118  raw=118
OK   q090 T3  gold=345  raw=345
OK   q091 T3  gold=188  raw=188
OK   q092 T3  gold=106  raw=106
OK   q105 T7  gold=417  raw=417


OK   q106 T7  gold=665  raw=665


OK   q107 T7  gold=235  raw=235
OK   q108 T7  gold=278  raw=278

Agree: 28   Disagree: 0


## 6. See a trap in action
For tricky questions, the look-alike table gives a **different, wrong number**, and nothing looks broken.
That's exactly what the agent has to avoid.

In [9]:
import duckdb
con = duckdb.connect("../data/ads.duckdb", read_only=True)

q = [x for x in questions if x["template"] == "T2"][0]
c = q["values"]["campaign"]
print(q["question"])
print("  correct (reach_campaign_total)        :", con.execute(f"SELECT unique_users FROM reach_campaign_total WHERE campaign_id = {c}").fetchone()[0])
print("  WRONG  (sum of reach_campaign_daily)  :", con.execute(f"SELECT SUM(unique_users) FROM reach_campaign_daily WHERE campaign_id = {c}").fetchone()[0])

q = [x for x in questions if x["template"] == "T1"][0]
c = q["values"]["campaign_suspicious"]
print("\n" + q["question"])
print("  correct (agg_campaign_daily_valid)    :", con.execute(f"SELECT ctr FROM agg_campaign_daily_valid WHERE campaign_id = {c} AND event_date = '2017-05-12'").fetchone()[0])
print("  WRONG  (agg_campaign_daily, all)      :", con.execute(f"SELECT ctr FROM agg_campaign_daily WHERE campaign_id = {c} AND event_date = '2017-05-12'").fetchone()[0])
con.close()

How many unique people saw campaign 331406 from 6 to 12 May?
  correct (reach_campaign_total)        : 1821
  WRONG  (sum of reach_campaign_daily)  : 1969

What was campaign 143747's CTR yesterday, excluding suspicious clicks?
  correct (agg_campaign_daily_valid)    : 0.056338028169014086
  WRONG  (agg_campaign_daily, all)      : 0.0625


## 7. The follow-up conversations (memory test)
Turn 2 and turn 3 only make sense if the system remembers turn 1 (*"split **that**"*, *"**it**"*, *"**this** campaign"*).

In [10]:
conversations = load_jsonl("../benchmark/conversations.jsonl")
for c in conversations[:4]:
    print(f"{c['id']} ({c['split']}, template {c['template']})")
    for k, turn in enumerate(c["turns"], start=1):
        print(f"   {k}. {turn['question']:60s} -> {str(gold[c['id'] + '_t' + str(k)])[:60]}")

c01 (practice, template C1)
   1. What was campaign 230181's CTR yesterday?                    -> [[0.1056338028169014]]
   2. Now split that by age level.                                 -> [['2', 0.061224489795918366], ['3', 0.136986301369863], ['4'
   3. And what was the overall CTR the day before?                 -> [[0.09141274238227147]]
c02 (test, template C1)
   1. What was campaign 187911's CTR yesterday?                    -> [[0.04580152671755725]]
   2. Now split that by age level.                                 -> [['1', 0.0], ['2', 0.03508771929824561], ['3', 0.05232558139
   3. And what was the overall CTR the day before?                 -> [[0.06923076923076923]]
c03 (test, template C1)
   1. What was campaign 55121's CTR yesterday?                     -> [[0.05076628352490421]]
   2. Now split that by age level.                                 -> [['0', 0.0], ['1', 0.051470588235294115], ['2', 0.0673076923
   3. And what was the overall CTR the day before?            

## 8. Your review, then freeze 🔒
**Be honest in the README:** these questions were drafted with AI help (Claude) from templates I designed and
reviewed, and every answer comes from running real SQL on the real data. Nothing is made up. That's allowed;
hiding it isn't.

**Before freezing, please:**
1. Read `questions.jsonl` (or section 3 above): do the questions sound like something an ad team would ask?
2. Hand-check about 10 answers yourself, e.g. run a gold SQL in a notebook and compare with `gold_results.jsonl`.
3. *(Optional, from the design doc)* ask a classmate to write ~20 questions without seeing the system.

**Then freeze**, from the project folder in the terminal:
```
.venv/bin/python benchmark/check_gold.py --freeze
```
This writes `benchmark/FROZEN.txt` with a fingerprint. After that, `make_benchmark.py` refuses to run again,
and any later change to a question changes the fingerprint, so it's visible.

**Next step (M4):** versions A and B + the test harness that scores them on the practice questions.